In [ ]:
import pandas as pd

# Read files
ananya_df = pd.read_csv("final_edges_with_fdr.csv",header=None,dtype=str,low_memory=False)
# Remove last two characters (-P)
ananya_df.iloc[1:, 0] = ananya_df.iloc[1:, 0].str.slice(stop=-2)
lauren_df = pd.read_csv("lauren_fdr_table.csv",header=None)

# Strip of whitespace
ananya_df = ananya_df.apply(lambda col: col.str.strip())
lauren_df = lauren_df.apply(lambda col: col.str.strip())
#print(ananya_df.head())
#print(lauren_df.head())

# Merge gene-metabolite pair based on (gene, metabolite)
merged = ananya_df.merge(lauren_df,on=[0, 1],how="inner",suffixes=("_ananya", "_lauren"))


# NaN-safe comparison function
def nan_equal(a, b):
    return (a == b) | (pd.isna(a) & pd.isna(b))
# Create comparison dataframe
comparison = pd.DataFrame({
    "gene": merged[0],
    "metabolite": merged[1],
    "Pooled r": nan_equal(merged[11], merged['2_lauren']),
    "Pooled p": nan_equal(merged[12], merged['3_lauren']),
    "Pooled FDR": nan_equal(merged[16], merged['4_lauren']),
})



diff_rows = comparison[
    ~(comparison[["Pooled r", "Pooled p", "Pooled FDR"]].all(axis=1))
].copy()
# Ananya vs lauren values side by side
detailed_diff = pd.DataFrame({
    "gene": diff_rows["gene"],
    "metabolite": diff_rows["metabolite"],

    "Pooled r (ananya)": merged.loc[diff_rows.index, 11],
    "Pooled r (lauren)": merged.loc[diff_rows.index, "2_lauren"],

    "Pooled p (ananya)": merged.loc[diff_rows.index, 12],
    "Pooled p (lauren)": merged.loc[diff_rows.index, "3_lauren"],

    "Pooled FDR (ananya)": merged.loc[diff_rows.index, 16],
    "Pooled FDR (lauren)": merged.loc[diff_rows.index, "4_lauren"],
})

